# TrendLens — 04 · Temporal Trends (Phase 4)

Stage 9 + Stage 11: per-cluster temporal aggregation, growth metrics, the emerging-trend score (documented + compared against alternatives), lifecycle classification, and the text baseline.

> **Integrity:** timestamps are the **neutral synthetic** ones (no rigged lifecycles). On this data the counts per cluster per month are tiny, so the growth signal is **noise-dominated** — an honest finding that real timestamps are required for real trend claims.

In [1]:
import sys
from pathlib import Path
REPO = Path.cwd()
if not (REPO / "config.py").exists():
    for p in Path.cwd().parents:
        if (p / "config.py").exists():
            REPO = p; break
sys.path.insert(0, str(REPO))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src import clustering, trends

In [2]:
meta = clustering.load_aligned_metadata()
labels = np.load(config.CLUSTER_MODELS_DIR / "labels_umap10.npy")
df = meta.copy()
df["cluster_id"] = labels
df = df[df["cluster_id"] >= 0].copy()
print("clustered posts:", len(df), "| clusters:", df["cluster_id"].nunique())

clustered posts: 3678 | clusters: 29


## 1 · Aggregate posts per (cluster, month)

In [3]:
agg = trends.aggregate_cluster_trends(df, period="M")
agg.head()

,cluster_id,period,post_count,unique_users,average_engagement,median_engagement
0,0,2010-01,2,2,75.5,75.5
1,0,2010-03,2,2,55.0,55.0
2,0,2010-06,1,1,59.0,59.0
3,0,2010-07,1,1,70.0,70.0
4,0,2010-09,1,1,76.0,76.0


## 2 · Growth metrics

In [4]:
metrics = trends.growth_metrics(agg, window=3)
metrics.head()

,cluster_id,n_posts,mean_period_posts,recent_growth,percentage_growth,slope,rolling_growth,acceleration,median_engagement,average_engagement
0,0,168,1.411765,0.6,0.333333,0.002614,0.027778,0.023810,49.50,71.174390
1,1,170,1.416667,1.0,0.000000,0.001264,0.166667,0.205128,47.25,84.527436
2,2,53,0.445378,1.0,0.000000,-0.000306,0.333333,0.115385,41.75,81.713768
3,3,66,0.573913,2.0,-0.333333,-0.002935,0.333333,-0.111111,46.00,76.733333
4,4,77,0.663793,0.0,0.500000,-0.000496,0.666667,-0.025000,49.00,70.828869


## 3 · Emerging trend score (documented formula)

Primary score **S3 = norm(recent_growth) × norm(log-size) × norm(stability)**.

Alternatives for comparison: S1 (growth only), S2 (growth × size). A large-but-flat cluster must never outrank a small-but-growing one — growth is the driver.

In [5]:
scored = trends.trend_scores(metrics)
ranked = trends.classify_lifecycle(scored)
ranked.sort_values("trend_score_growth_size_stability", ascending=False).head(10)

,cluster_id,n_posts,mean_period_posts,recent_growth,percentage_growth,slope,rolling_growth,acceleration,median_engagement,average_engagement,trend_score_growth,trend_score_growth_size,trend_score_growth_size_stability,lifecycle
21,21,218,1.816667,1.400000,1.500000,0.000854,0.444444,0.018349,46.00,64.911699,0.520000,0.381025,0.199351,Rising
25,25,369,3.075000,0.125000,-0.200000,-0.008469,1.305556,-0.077720,53.50,86.633642,0.137500,0.137500,0.137500,Stable
16,16,212,1.781513,0.750000,-0.500000,-0.003960,-0.055556,-0.155172,54.00,75.400000,0.325000,0.233540,0.119076,Rising
20,20,325,2.708333,0.111111,0.666667,-0.002115,0.072222,-0.087719,47.25,79.041061,0.133333,0.124731,0.107401,Stable
1,1,170,1.416667,1.000000,0.000000,0.001264,0.166667,0.205128,47.25,84.527436,0.400000,0.242663,0.090181,Rising
0,0,168,1.411765,0.600000,0.333333,0.002614,0.027778,0.023810,49.50,71.174390,0.280000,0.168185,0.062190,Rising
23,23,133,1.108333,1.333333,0.500000,0.000823,0.666667,-0.071429,48.50,73.551587,0.500000,0.241201,0.061457,Rising
28,28,182,1.516667,0.200000,-0.333333,-0.002799,0.500000,-0.160000,53.00,79.803125,0.160000,0.102595,0.042015,Stable
7,7,256,2.133333,-0.166667,-0.600000,-0.001070,-0.055556,0.031496,55.00,93.834455,0.050000,0.040714,0.026187,Stable
17,17,115,0.958333,0.333333,-0.600000,-0.001226,0.555556,0.017241,47.50,83.219953,0.200000,0.081778,0.016189,Rising


## 4 · Do the score alternatives agree? (Spearman)

In [6]:
cmp = trends.compare_score_alternatives(ranked)
pd.DataFrame(cmp).T

,spearman,p
trend_score_growth_vs_trend_score_growth_size,0.632209,2.339917e-04
trend_score_growth_vs_trend_score_growth_size_stability,0.274764,1.491616e-01
trend_score_growth_size_vs_trend_score_growth_size_stability,0.895430,5.465117e-11


## 5 · Lifecycle classification

`recent_growth` thresholds → Rising / Stable / Declining. **Honest caveat:** with ~0.5–2 posts per cluster per month and a 3-month window, most variation is small-count noise — treat the lifecycle spread as a demonstration, not a detection.

In [7]:
ranked["lifecycle"].value_counts()

lifecycle
Rising       15
Stable       10
Declining     4
Name: count, dtype: int64

## 6 · Text baseline (Stage 11, v1 — tags only)

For each cluster, dominant tags' growth across the whole dataset → a per-cluster text trend score. Will feed the detection lead-time experiment (Phase 8).

In [8]:
text_scores = trends.text_trend_scores(df, period="M")
merged = ranked.merge(text_scores, on="cluster_id", how="left")
merged[["cluster_id", "recent_growth", "lifecycle", "text_trend_score"]].head(8)

,cluster_id,recent_growth,lifecycle,text_trend_score
0,0,0.600000,Rising,0.490693
1,1,1.000000,Rising,0.473096
2,2,1.000000,Rising,0.618225
3,3,2.000000,Rising,0.223922
4,4,0.000000,Stable,0.334705
5,5,-0.142857,Stable,0.478450
6,6,2.000000,Rising,0.412923
7,7,-0.166667,Stable,0.388066


## 7 · Charts for top trends

In [9]:
paths = trends.plot_cluster_trends(agg, scored, top_n=4)
print([p.name for p in paths])

['trend_cluster_021.png', 'trend_cluster_025.png', 'trend_cluster_016.png', 'trend_cluster_020.png']


## Phase 4 checkpoint
- [x] Per-cluster weekly/monthly aggregation
- [x] Growth metrics + documented trend score (3 alternatives compared)
- [x] Lifecycle classification
- [x] Text baseline (tag growth)
- [x] Charts + `trend_metrics.csv` + experiment manifest

**Next (Phase 5):** cluster interpretation — VLM captions on representative images → names/descriptions (clearly labelled as interpretation).